In [5]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import glob
from pathlib import Path
import os
import zipfile
import os
from shapely.geometry import Point

In [6]:
# Nepal Map Shape file dataset
zip_path = "nepal_shape/npl_admin_boundaries.shp.zip"
extract_path = "nepal_shape/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [7]:
nepal_map = gpd.read_file("nepal_shape/npl_admin2.shp")
# nepal_map['adm2_name', 'adm1_name', 'adm1_pcode', 'area_sqkm', 'cod_versio', 'center_lat', 'center_lon', 'geometry']
nepal_map.head()

,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,valid_to,area_sqkm,cod_versio,lang,lang1,lang2,lang3,center_lat,center_lon,geometry
0,Taplejung,None,None,None,NP0101,Koshi,None,None,None,NP01,...,NaT,3631.813309,V_02,en,None,None,None,27.610463,87.821542,"POLYGON ((87.83403 27.95158, 87.83465 27.95139..."
1,Panchthar,None,None,None,NP0102,Koshi,None,None,None,NP01,...,NaT,1247.722512,V_02,en,None,None,None,27.148871,87.855072,"POLYGON ((88.07442 27.43312, 88.07464 27.43312..."
2,Ilam,None,None,None,NP0103,Koshi,None,None,None,NP01,...,NaT,1685.282883,V_02,en,None,None,None,26.885396,87.920240,"POLYGON ((87.95472 27.10455, 87.95545 27.10444..."
3,Jhapa,None,None,None,NP0104,Koshi,None,None,None,NP01,...,NaT,1603.810951,V_02,en,None,None,None,26.583568,87.885501,"POLYGON ((88.15222 26.80584, 88.1528 26.80534,..."
4,Morang,None,None,None,NP0105,Koshi,None,None,None,NP01,...,NaT,1822.340361,V_02,en,None,None,None,26.610614,87.473116,"POLYGON ((87.50698 26.8704, 87.50738 26.86958,..."


In [8]:
# Overall Fire incidents dataset: 2020-2024
shp_files = list(Path("data/Fire Incidents/").glob("*/*.shp"))

shp_files = [f for f in shp_files if not f.name.startswith("._")]

fire_data_list = []

for file in shp_files:
    try:
        gdf = gpd.read_file(file)
        
        year = file.parent.name
        gdf['year'] = int(year)
        
        fire_data_list.append(gdf)
    
    except Exception as e:
        print(f"Skipping file: {file} → {e}")

fire_data = gpd.GeoDataFrame(pd.concat(fire_data_list, ignore_index=True))

fire_data.head()

,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,INSTRUMENT,CONFIDENCE,VERSION,BRIGHT_T31,FRP,DAYNIGHT,TYPE,geometry,year
0,26.90225,85.80494,328.11,0.40,0.37,2020-01-02,0729,N20,VIIRS,n,2,290.10,2.20,D,0,POINT (85.80494 26.90225),2020
1,27.47337,84.33176,328.18,0.38,0.36,2020-01-02,0729,N20,VIIRS,n,2,291.14,2.03,D,0,POINT (84.33176 27.47337),2020
2,27.47309,82.85561,327.18,0.40,0.37,2020-01-02,0729,N20,VIIRS,n,2,293.35,2.15,D,0,POINT (82.85561 27.47309),2020
3,28.77123,80.20963,337.18,0.49,0.49,2020-01-05,0814,N20,VIIRS,n,2,290.54,6.05,D,0,POINT (80.20963 28.77123),2020
4,27.62135,83.08981,295.71,0.42,0.38,2020-01-06,2020,N20,VIIRS,n,2,277.86,0.42,N,0,POINT (83.08981 27.62135),2020


In [9]:
# Nepal's Forest Cover Dataset: 2020-2024
file_path = "data/Forest Cover.xlsx"
forest_cover_data = pd.read_excel(file_path)
forest_cover_data.head()

,S.NO,District,mean_slope_percent,Population Density,Road Density (Km/100 Km2),Rainfall 2020 mm,Forest Cover,Year
0,1,ACHHAM,55.410044,136.221429,7.51,1334.4,2.326786,2020
1,2,ARGHAKHANCHI,46.532218,148.437552,24.43,1775.4,0.568315,2020
2,3,BAGLUNG,62.384511,139.692265,12.55,2011.0,1.299327,2020
3,4,BAITADI,58.269744,159.418697,10.78,1501.8,0.809085,2020
4,5,BAJHANG,69.153910,55.255698,3.17,1366.0,5.914670,2020


In [10]:
import rasterio

In [11]:
raster_paths = {
    2020: "data/Land Cover/2020/lc2020.tif",
    2021: "data/Land Cover/2021/lc2021.tif",
    2022: "data/Land Cover/2022/lc2022.tif" 
}

In [12]:
# For years 2023, 2024, use the 2022 raster as proxy
def get_raster_path(year):
    if year in raster_paths:
        return raster_paths[year]
    else:
        # Fallback for 2023, 2024, or any missing year
        return raster_paths[2022]

In [13]:
fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])
fire_data['year'] = fire_data['ACQ_DATE'].dt.year

In [14]:
def get_forest_value(point, year):
    """Extract forest class/value at point for given year."""
    # Use 2022 as proxy for 2023, 2024, or missing years
    raster_year = year if year in raster_paths else 2022
    path = raster_paths[raster_year]
    with rasterio.open(path) as src:
        # Reproject point to raster CRS (assuming fires_gdf in EPSG:4326)
        point_proj = gpd.GeoSeries([point], crs="EPSG:4326").to_crs(src.crs).iloc[0]
        val = next(src.sample([(point_proj.x, point_proj.y)]))[0]
    return val

In [17]:
# Apply to each fire
fire_data['forest_value'] = fire_data.apply(
    lambda row: get_forest_value(row.geometry, row['year']), axis=1
)

In [18]:
FOREST_CLASS = 1   # change based on your data
fires_forest = fire_data[fire_data['forest_value'] == FOREST_CLASS].copy()
print(f"Original fires: {len(fire_data)}")
print(f"Forest fires: {len(fires_forest)}")

Original fires: 190071
Forest fires: 265


In [19]:
districts = nepal_map
districts = districts.to_crs("EPSG:4326")

# Spatial join to assign district to each forest fire
fires_with_district = gpd.sjoin(fires_forest, districts[['adm2_name', 'geometry']],
                                how='left', predicate='within')
fires_with_district = fires_with_district.rename(columns={'adm2_name': 'District'})

# Merge features (by District and year)
fires_model = fires_with_district.merge(forest_cover_data, on=['District', 'year'], how='left')

KeyError: 'year'

In [ ]:
# For rows where features are missing (likely 2023/2024), use 2022 values
features_cols = ['mean_slope_percent', 'Population Density', 
                 'Road Density (Km/100 Km2)', 'Rainfall 2020 mm', 'Forest Cover']
# Get 2022 features
feat_2022 = forest_cover_data[forest_cover_data['year'] == 2022].drop(columns='year')
# For each missing row, fill from 2022
for col in features_cols:
    fires_model[col] = fires_model.groupby('District')[col].apply(
        lambda x: x.fillna(feat_2022.set_index('District').loc[x.index.get_level_values(0), col] if len(x) > 0 else x)
    )

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor, GaussianProcessClassifier
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, RBF
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import matplotlib.pyplot as plt

# Prepare feature matrix and target
feature_cols = ['month', 'mean_slope_percent', 'Population Density',
                'Road Density (Km/100 Km2)', 'Rainfall 2020 mm', 'Forest Cover']
target = 'FRP'

# Drop rows with missing values
data = fires_model[feature_cols + [target]].dropna()
X = data[feature_cols].values
y = data[target].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- Regression ---
kernel = Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gpr.fit(X_train, y_train)
y_pred, y_std = gpr.predict(X_test, return_std=True)
print(f"GPR R²: {r2_score(y_test, y_pred):.3f}, MAE: {mean_absolute_error(y_test, y_pred):.2f}")

# --- Classification (binary) ---
threshold = np.median(y)
y_class = (y > threshold).astype(int)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_class, test_size=0.2, random_state=42)
gpc = GaussianProcessClassifier(kernel=1.0*RBF(), n_restarts_optimizer=5)
gpc.fit(X_train_c, y_train_c)
y_pred_c = gpc.predict(X_test_c)
print(f"GPC Accuracy: {accuracy_score(y_test_c, y_pred_c):.3f} (threshold FRP > {threshold:.2f})")

In [ ]:
# Predict on all fires (or aggregate by district)
fires_model['predicted_intensity'] = gpr.predict(scaler.transform(fires_model[feature_cols]))
district_summary = fires_model.groupby('District')['predicted_intensity'].mean().reset_index()
district_map = districts.merge(district_summary, left_on='adm2_name', right_on='District', how='left')
district_map.plot(column='predicted_intensity', cmap='Reds', legend=True,
                  legend_kwds={'label': 'Predicted Fire Intensity (FRP)'})
plt.title('Forest Fire Intensity Prediction by District (2020-2024)')
plt.show()